# Step 11.4 — Scaling and Softmax：为什么要除以 √d_k，再把 Scores 变成 Attention Weights？

Step 11.3 已经得到 raw compatibility scores：

$$
S=QK^T
\in
\mathbb R^{B\times T\times T}.
$$

其中：

$$
S_{i,j}=q_i\cdot k_j.
$$

现在还剩两个问题：

1. 当 `d_k` 变大时，dot-product scores 的典型幅度为什么容易变大？
2. raw scores 怎样变成每个 Query position 对所有 Keys 的 normalized attention weights？

标准 scaled dot-product Attention 的回答是：

```text
QK^T
  ↓ divide by sqrt(d_k)
scaled scores
  ↓ softmax over Key axis
attention weights A
```

本节只到这里；不进入 Step 11.5，也不计算 `A @ V`。


## 1. 为什么 d_k 变大时，Dot Product 的典型尺度容易变大？

单个 Query–Key score 是：

$$
s=q\cdot k
=
\sum_{m=1}^{d_k}q_mk_m.
$$

为了建立 scaling 的统计直觉，先使用一个**教学假设**：

```text
q_m 与 k_m 彼此独立
每个分量均值约为 0
每个分量方差约为 1
```

在这个简化假设下，单项：

$$
z_m=q_mk_m
$$

有近似：

$$
\mathbb E[z_m]=0,
\qquad
\mathrm{Var}(z_m)=1.
$$

而 score 是 d_k 个这样的项相加：

$$
s=\sum_{m=1}^{d_k}z_m.
$$

如果这些项近似独立，则方差相加：

$$
\boxed{
\mathrm{Var}(s)
\approx
d_k
}
$$

因此 standard deviation（标准差）约为：

$$
\boxed{
\mathrm{Std}(s)
\approx
\sqrt{d_k}
}
$$

这就是为什么 feature width 增大时，raw dot products 的典型幅度容易一起增大。

重要边界：真实训练后的 Q/K components 不一定独立、零均值、单位方差，所以这里是解释 `1/√d_k` 的经典统计动机，不是对任意训练状态的精确概率定律。


## 2. 为什么正好除以 √d_k？

如果：

$$
\mathrm{Var}(s)\approx d_k,
$$

定义 scaled score：

$$
\tilde s
=
\frac{s}{\sqrt{d_k}}.
$$

利用：

$$
\mathrm{Var}(aX)=a^2\mathrm{Var}(X),
$$

得到：

$$
\mathrm{Var}(\tilde s)
=
\frac{1}{d_k}\mathrm{Var}(s)
\approx
1.
$$

所以：

$$
\boxed{
\tilde S
=
\frac{QK^T}{\sqrt{d_k}}
}
$$

主要作用不是改变 score matrix 的 shape，也不是把 scores 归一化到 `[0,1]`，而是让不同 `d_k` 下 score 的典型尺度更稳定。

Shape 完全不变：

$$
(B,T,T)
\rightarrow
(B,T,T).
$$

### 为什么这对 Softmax 有帮助？

如果一行 logits 的数值差距很大，softmax 会变得非常 peaked（尖锐）：最大的几个位置接近 1，其余位置接近 0。

过大的 raw dot products 因此更容易把 softmax 推到非常尖锐的区域。Scaling 让 logits 的典型幅度更温和，使后续 normalization 在训练早期通常更容易优化。

这不是保证 attention 永远均匀，也不是说大 score 一定错误；模型仍然可以学习产生非常集中的 attention。Scaling 只是控制进入 softmax 前的典型数值尺度。


## 3. 一个小例子：同样的相对顺序，Scaling 会让 Softmax 没那么尖

假设某个 Query position 对三个 Keys 的 raw scores 是：

$$
[4,2,0].
$$

并假设：

$$
d_k=4.
$$

所以：

$$
\sqrt{d_k}=2.
$$

scaled scores：

$$
\frac{[4,2,0]}{2}
=
[2,1,0].
$$

注意 scaling 没有改变大小顺序：

```text
Key 0 > Key 1 > Key 2
```

但数值间距缩小了。

对 raw `[4,2,0]` 做 softmax，大约得到：

$$
[0.867,0.117,0.016].
$$

对 scaled `[2,1,0]` 做 softmax，大约得到：

$$
[0.665,0.245,0.090].
$$

两者都认为 Key 0 最相关，但 scaled 版本没有那么极端。

这就是 scaling 的直觉：

> **保留 score 的相对结构，同时控制进入 softmax 的典型幅度。**


In [ ]:
import math
import mlx.core as mx

raw_scores = mx.array([4.0, 2.0, 0.0])
d_k = 4

scale = math.sqrt(d_k)
scaled_scores = raw_scores / scale

raw_weights = mx.softmax(raw_scores, axis=-1)
scaled_weights = mx.softmax(scaled_scores, axis=-1)

print('sqrt(d_k):', scale)
print('raw scores:', raw_scores)
print('scaled scores:', scaled_scores)
print('softmax(raw):', raw_weights)
print('softmax(scaled):', scaled_weights)
print('scaled weight sum:', mx.sum(scaled_weights))


## 4. 对 T×T Score Matrix，Softmax 到底沿哪个 Axis？

Step 11.3 已经确定：

```text
rows    = Query positions i
columns = Key positions j
```

所以对固定 Query position `i`，我们要把这一行：

$$
S_{i,:}
$$

转换成“它对所有 Keys 的权重分布”。

因此 softmax 必须沿 **Key axis（列方向）**。

Batch 下 scaled scores：

$$
\tilde S
\in
\mathbb R^{B\times T\times T}.
$$

最后一个 axis 正好就是 Key-position axis，所以 MLX 写：

```python
A = mx.softmax(scaled_scores, axis=-1)
```

输出：

$$
A\in\mathbb R^{B\times T\times T}.
$$

Shape 不变，但每个 Query row 现在满足：

$$
A_{i,j}\ge0
$$

以及：

$$
\boxed{
\sum_{j=0}^{T-1}A_{i,j}=1
}
$$

因此：

```text
A[b, i, :]
= sample b 中 Query position i
  对所有 Key positions 的 attention distribution
```

如果错误地沿 Query axis 做 softmax，就会把“所有 Queries 对同一个 Key”的分数归一化，这不是当前 Self-Attention retrieval 所需要的方向。


In [ ]:
B, T, d_k = 2, 3, 4

# Illustrative raw QK^T score matrices.
raw_scores = mx.array([
    [[4.0, 2.0, 0.0],
     [1.0, 3.0, 2.0],
     [0.0, 1.0, 2.0]],

    [[2.0, 0.0, 1.0],
     [3.0, 1.0, 0.0],
     [1.0, 1.0, 1.0]],
])  # (B,T,T)

scaled_scores = raw_scores / math.sqrt(d_k)  # (B,T,T)
attention_weights = mx.softmax(scaled_scores, axis=-1)  # (B,T,T)

print('raw_scores:', raw_scores.shape)
print('scaled_scores:', scaled_scores.shape)
print('attention_weights:', attention_weights.shape)
print('row sums:', mx.sum(attention_weights, axis=-1))


## 5. 到这里，Self-Attention 的“权重计算部分”第一次完整了

从 Step 11.2 开始：

$$
Q=XW_Q,
\qquad
K=XW_K.
$$

Step 11.3 得到 raw scores：

$$
S=QK^T.
$$

Step 11.4 先 scaling：

$$
\tilde S
=
\frac{QK^T}{\sqrt{d_k}}.
$$

再沿每个 Query row 的 Key axis 做 softmax：

$$
\boxed{
A
=
\operatorname{softmax}
\left(
\frac{QK^T}{\sqrt{d_k}}
\right)
}
$$

Shape flow：

```text
Q                 : (B,T,d_k)
K                 : (B,T,d_k)
QK^T              : (B,T,T)
scaled scores     : (B,T,T)
attention weights : (B,T,T)
```

其中：

```text
A[b,i,j]
= sample b 中
  Query position i
  给 Key position j 的 normalized attention weight
```

### Numerical-stability note

Softmax 的数学定义包含 exponentials；稳定实现通常会避免直接对非常大的 logits 做 naive exponentiation。MLX 的 `mx.softmax` 应作为框架操作直接使用，而不是在训练代码里手写朴素 `exp(x)/sum(exp(x))`。

本节只需要知道这个工程边界，不展开 softmax kernel 或 stable-softmax 推导。

### Step 11.4 材料边界

- 在独立、零均值、单位方差的教学假设下，dot-product score 的方差约为 `d_k`、标准差约为 `sqrt(d_k)`；这是 scaling 的经典统计动机，不是对训练后 Q/K 的绝对定律。
- scaled score 是 `QK^T / sqrt(d_k)`；scaling 不改变 `(B,T,T)` shape。
- scaling 主要控制进入 softmax 前的典型 logit 幅度，不负责把数值变成概率。
- softmax 沿 Key axis 做 normalization；对 `(B,T,T)` 使用 `axis=-1`。
- `A[b,i,:]` 是 Query position i 对所有 Keys 的非负、和为 1 的 attention distribution。
- 到本节为止只得到 attention weights A；还没有把它们用于 Value retrieval。
- 本轮只准备 Step 11.4；不进入 Step 11.5，不计算 `A @ V`。
